# Ray Unit 2 - MapReduce Word Count

In this notebook we'll implement the **MapReduce** distributed programming model with Ray and use it as a testbed for a few advanced **execution control patterns**.

Our program will solve the **word count** problem:
1. **Input:** a text corpus.
2. **Output:** a list of all words in the corpus and their number of occurrences (their **counts**).

## Prerequisites

Understand the MapReduce workflow applied to the word count problem:
1. **Initialization:** the driver chunks data and distributes chunks to cluster workers.
2. **Map phase:** workers count words in their respective chunks and produce pairs of `(word, count)`.
3. **Global shuffle:** workers communicate globally and transfer data such that all `(X, count)` for a given word `X` are localized on a single worker.
4. **Reduce phase:** workers sum the counts of all localized words to produce pairs of `(word, sum(counts))`.
5. **Finalization:** workers return their respective reduce outputs to the driver, and the driver outputs the full word-count list.

This notebook starts from that classical model and then deliberately simplifies part of the reduction path so we can isolate coordination behavior.

For more details, see: [Wikipedia](https://en.wikipedia.org/wiki/MapReduce), [Ray docs](https://docs.ray.io/en/latest/ray-core/examples/map_reduce.html).

## Setup

Activate and run this notebook with the `22971-ray` environment.
The next cell starts a local Ray runtime with the dashboard disabled so the setup stays quiet and the notebook stays focused on the execution patterns.

In [1]:
import time
import ray

from collections import Counter

/Users/shlomibenshushan/miniconda3/envs/22971-ray/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-19 07:23:50,473	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
ray.shutdown()
ray.init(include_dashboard=False)
ray.cluster_resources()

2026-04-19 07:23:53,856	INFO worker.py:2013 -- Started a local Ray instance.
/Users/shlomibenshushan/miniconda3/envs/22971-ray/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


{'object_store_memory': 2147483648.0,
 'node:__internal_head__': 1.0,
 'node:127.0.0.1': 1.0,
 'memory': 9072115712.0,
 'CPU': 12.0}

Our data is this list of random sentences:

In [3]:
BASE_DOCS = [
    "winter markets opened before dawn beside the river station".split(),
    "three interns traded sandwich halves while the build pipeline stalled".split(),
    "an orange bicycle leaned against the bakery during light rain".split(),
    "the archive room smelled like paper dust and warm plastic".split(),
    "neighbors compared telescope photos after clouds drifted past midnight".split(),
    "a sleepy cashier counted lemons twice and still shrugged".split(),
    "someone left jazz records near the elevator with a handwritten note".split(),
    "our test cluster blinked green while coffee cooled on the desk".split(),
    "late ferries carried commuters home under a pale harbor moon".split(),
    "the workshop window rattled whenever delivery trucks crossed the bridge".split(),
]

After running MapReduce, we expect to get back this output:

In [4]:
expected_output =  Counter({
    'the': 8, 'a': 3, 'while': 2, 'and': 2, 'winter': 1, 'markets': 1, 'opened': 1, 'before': 1, 'dawn': 1, 'beside': 1,
    'river': 1, 'station': 1, 'three': 1, 'interns': 1, 'traded': 1, 'sandwich': 1, 'halves': 1, 'build': 1,
    'pipeline': 1, 'stalled': 1, 'an': 1, 'orange': 1, 'bicycle': 1, 'leaned': 1, 'against': 1, 'bakery': 1,
    'during': 1, 'light': 1, 'rain': 1, 'archive': 1, 'room': 1, 'smelled': 1, 'like': 1, 'paper': 1, 'dust': 1,
    'warm': 1, 'plastic': 1, 'neighbors': 1, 'compared': 1, 'telescope': 1, 'photos': 1, 'after': 1, 'clouds': 1,
    'drifted': 1, 'past': 1, 'midnight': 1, 'sleepy': 1, 'cashier': 1, 'counted': 1, 'lemons': 1, 'twice': 1,
    'still': 1, 'shrugged': 1, 'someone': 1, 'left': 1, 'jazz': 1, 'records': 1, 'near': 1, 'elevator': 1, 'with': 1,
    'handwritten': 1, 'note': 1, 'our': 1, 'test': 1, 'cluster': 1, 'blinked': 1, 'green': 1, 'coffee': 1, 'cooled': 1,
    'on': 1, 'desk': 1, 'late': 1, 'ferries': 1, 'carried': 1, 'commuters': 1, 'home': 1, 'under': 1, 'pale': 1,
    'harbor': 1, 'moon': 1, 'workshop': 1, 'window': 1, 'rattled': 1, 'whenever': 1, 'delivery': 1, 'trucks': 1,
    'crossed': 1, 'bridge': 1
})
expected_output.most_common(5)

[('the', 8), ('a', 3), ('while', 2), ('and', 2), ('winter', 1)]

## Helper functions

In [5]:
def format_top(counter: Counter[str], top_k: int = 5) -> str:
    return ", ".join(f"{word}:{count}" for word, count in counter.most_common(top_k))

format_top(expected_output)

'the:8, a:3, while:2, and:2, winter:1'

In [6]:
def flatten_tokens(docs: list[list[str]]) -> list[str]:
    return [token for doc in docs for token in doc]

print(BASE_DOCS[0:2])
print(flatten_tokens(BASE_DOCS[0:2]))

[['winter', 'markets', 'opened', 'before', 'dawn', 'beside', 'the', 'river', 'station'], ['three', 'interns', 'traded', 'sandwich', 'halves', 'while', 'the', 'build', 'pipeline', 'stalled']]
['winter', 'markets', 'opened', 'before', 'dawn', 'beside', 'the', 'river', 'station', 'three', 'interns', 'traded', 'sandwich', 'halves', 'while', 'the', 'build', 'pipeline', 'stalled']


## 0. Initialization

Split the corpus into chunks

In [7]:
docs_per_chunk = 3

def chunked(items: list[list[str]], chunk_size: int) -> list[list[list[str]]]:
    return [items[i:i + chunk_size] for i in range(0, len(items), chunk_size)]

chunks = chunked(BASE_DOCS, docs_per_chunk)
print(chunks[0])
print(chunks[-1])
print(len(chunks[0]),len(chunks[-1]))

[['winter', 'markets', 'opened', 'before', 'dawn', 'beside', 'the', 'river', 'station'], ['three', 'interns', 'traded', 'sandwich', 'halves', 'while', 'the', 'build', 'pipeline', 'stalled'], ['an', 'orange', 'bicycle', 'leaned', 'against', 'the', 'bakery', 'during', 'light', 'rain']]
[['the', 'workshop', 'window', 'rattled', 'whenever', 'delivery', 'trucks', 'crossed', 'the', 'bridge']]
3 1


Launch one task per chunk (for now - we just move the data)

In [8]:
@ray.remote
def distribute_data(chunk_id: int, docs: list[list[str]]) -> dict:
    tokens = flatten_tokens(docs)
    return {
        "chunk_id": chunk_id,
        "num_docs": len(docs),
    }

print(f"Launching {len(chunks)} tasks...\n")

refs = [distribute_data.remote(chunk_id, docs) for chunk_id, docs in enumerate(chunks)]
results = ray.get(refs)

for result in results:
    print(f"chunk {result['chunk_id']:>2} | docs={result['num_docs']:>2}")

Launching 4 tasks...

chunk  0 | docs= 3
chunk  1 | docs= 3
chunk  2 | docs= 3
chunk  3 | docs= 1


## 1. Map
We modify the remote function to produce word counts of their chunks

In [9]:
@ray.remote
def count_words_in_chunk(chunk_id: int, docs: list[list[str]]) -> dict:
    tokens = flatten_tokens(docs)
    counts = Counter(tokens)
    return {
        "chunk_id": chunk_id,
        "num_docs": len(docs),
        "num_tokens": len(tokens),
        "counts": counts,
    }

Copy `BASE_DOCS` a few times for a larger corpus (more chunks `=>` more tasks) and launch tasks.

In [10]:
repeat = 20
corpus = BASE_DOCS * repeat
print(f"Corpus size: {len(corpus)} docs")

Corpus size: 200 docs


In [11]:
docs_per_chunk = 3
chunks = chunked(corpus, docs_per_chunk)
print(f"Tasks to process: {len(chunks)}")

Tasks to process: 67


In [12]:
refs = [count_words_in_chunk.remote(chunk_id, docs) for chunk_id, docs in enumerate(chunks)]
results = ray.get(refs)

for result in results:
    print(f"chunk {result['chunk_id']:>2} | docs ={result['num_docs']:>2} | "
          f"tokens ={result['num_tokens']:>4} | top words: {format_top(result['counts'], 3)}")

chunk  0 | docs = 3 | tokens =  29 | top words: the:3, winter:1, markets:1
chunk  1 | docs = 3 | tokens =  28 | top words: and:2, the:1, archive:1
chunk  2 | docs = 3 | tokens =  32 | top words: the:2, a:2, someone:1
chunk  3 | docs = 3 | tokens =  29 | top words: the:4, workshop:1, window:1
chunk  4 | docs = 3 | tokens =  29 | top words: the:2, an:1, orange:1
chunk  5 | docs = 3 | tokens =  31 | top words: a:2, the:2, sleepy:1
chunk  6 | docs = 3 | tokens =  29 | top words: the:3, late:1, ferries:1
chunk  7 | docs = 3 | tokens =  30 | top words: the:3, three:1, interns:1
chunk  8 | docs = 3 | tokens =  29 | top words: a:2, neighbors:1, compared:1
chunk  9 | docs = 3 | tokens =  31 | top words: the:3, our:1, test:1
chunk 10 | docs = 3 | tokens =  29 | top words: the:3, winter:1, markets:1
chunk 11 | docs = 3 | tokens =  28 | top words: and:2, the:1, archive:1
chunk 12 | docs = 3 | tokens =  32 | top words: the:2, a:2, someone:1
chunk 13 | docs = 3 | tokens =  29 | top words: the:4, wor

## 2. Evolving MapReduce

We'll write three versions of MapReduce, in increasing sophistication.

1. At the system design stage, we decide the number of reducers and the data range they'll reduce. We'll choose 3 reducers, with alphabetical division of labor
- **reducer 0:** words that start with a-i
- **reducer 1:** j-r
- **reducer 2:** s-z

2. We make the mappers aware of this division of labor and divide the return value into three buckets - one destined for each reducer.

3. The driver will collect mapper outputs, pass them on to the reducers directly as it launches the reduce tasks and finally join the reducer outputs.

### Version 1.0: driver shuffle

In [13]:
def reducer_for_word(word: str) -> int:
    first = word[0].lower()
    if first <= "i":
        return 0
    if first <= "r":
        return 1
    return 2

In [14]:
@ray.remote
def count_words_in_chunk_return_buckets(chunk_id: int, docs: list[list[str]]) -> dict:
    tokens = flatten_tokens(docs)
    buckets = {0: Counter(), 1: Counter(), 2: Counter()}

    for word in tokens:
        buckets[reducer_for_word(word)][word] += 1

    return {
        "chunk_id": chunk_id,
        "num_docs": len(docs),
        "num_tokens": len(tokens),
        "buckets": buckets,
    }

In [15]:
@ray.remote
def reduce_buckets(reducer_id: int, buckets: list[Counter[str]]) -> dict:
    counts = Counter()
    for bucket in buckets:
        counts.update(bucket)
    return {
        "reducer_id": reducer_id,
        "counts": counts,
    }

In [16]:
print(f"Launching {len(chunks)} tasks...")

# Map tasks produce buckets for the shuffle, but we don't launch any reduce tasks until the shuffle is complete.
map_output_refs = [
    count_words_in_chunk_return_buckets.remote(chunk_id, docs)
    for chunk_id, docs in enumerate(chunks)
]
map_outputs = ray.get(map_output_refs)

# The driver performs the shuffle by regrouping buckets per reducer
routed_buckets = {0: [], 1: [], 2: []}
for result in map_outputs:
    for reducer_id, bucket in result["buckets"].items():
        routed_buckets[reducer_id].append(bucket)

# Only after the full shuffle do we launch one reducer task per bucket group
reduce_output_refs = [
    reduce_buckets.remote(reducer_id, routed_buckets[reducer_id])
    for reducer_id in range(len(routed_buckets))
]
reduce_outputs = ray.get(reduce_output_refs)

# Display reducer outputs and verify global count matches direct count
print("\nReducer outputs:")
for result in reduce_outputs:
    print(f"reducer {result['reducer_id']} | unique_words ={len(result['counts']):>3} | "
          f"top words: {format_top(result['counts'], 4)}")

# Verify global count matches direct count by merging reducer outputs on the driver after the shuffle
total_counts_driver_shuffle = Counter()
for result in reduce_outputs:
    total_counts_driver_shuffle.update(result["counts"])

# Compare to direct count on the driver without shuffle
expected_counts = Counter(flatten_tokens(corpus))
print("\nGlobal word count (top words):")
print(format_top(total_counts_driver_shuffle, 12))
print(f"matches direct count = {total_counts_driver_shuffle == expected_counts}")

Launching 67 tasks...

Reducer outputs:
reducer 0 | unique_words = 37 | top words: a:60, and:40, before:20, dawn:20
reducer 1 | unique_words = 28 | top words: markets:20, opened:20, river:20, pipeline:20
reducer 2 | unique_words = 23 | top words: the:160, while:40, winter:20, station:20

Global word count (top words):
the:160, a:60, and:40, while:40, before:20, dawn:20, beside:20, interns:20, halves:20, build:20, an:20, bicycle:20
matches direct count = True


### Anti-pattern: data materialization at the driver
In this implementation, the driver:
1. Launches the mapper tasks and gets output refs back.
2. Calls `ray.get(map_output_refs)` and materializes all mapper outputs locally.
3. Regroups the buckets per reducer (shuffle).
4. Creates new objects when invoking `.remote(...)` on the reducers.

This is problematic for three reasons:
1. The driver blocks on `map_outputs = ray.get(map_output_refs)` until all mappers terminate.
2. The entire mapper output must be localized at the driver before the shuffle. MapReduce is specifically designed for cases where the data is too big for this.
3. The driver does not actually modify the bucket contents. Pulling the data and sending it back out creates unnecessary overhead.

**Solution:**
1. Mappers will return each bucket as a separate object in the store.
2. The driver will keep only the routing metadata and pass `ObjectRef`s directly to the reducers.

This way, the driver can start the shuffle as soon as the bucket `ObjectRefs` are available, and it never materializes the big intermediate data.
### Version 2.0: driver `ObjectRef` routing

In [17]:
BucketRefs = tuple[ray.ObjectRef[Counter[str]], ray.ObjectRef[Counter[str]], ray.ObjectRef[Counter[str]]]

@ray.remote(num_returns=3)
def count_words_in_chunk_return_bucket_refs(docs: list[list[str]]) -> BucketRefs:
    # New: each returned bucket stays in the object store as its own object.
    buckets = [Counter(), Counter(), Counter()]
    for word in flatten_tokens(docs):
        buckets[reducer_for_word(word)][word] += 1
    return tuple(buckets)

In [18]:
@ray.remote
def reduce_bucket_refs(reducer_id: int, *buckets: Counter[str]) -> dict:  # New: reducer takes bucket refs as separate arguments instead of a list of buckets.
    counts = Counter()
    for bucket in buckets:
        counts.update(bucket)
    return {
        "reducer_id": reducer_id,
        "counts": counts,
    }

In [19]:
print(f"Launching {len(chunks)} tasks...")

# Map tasks produce buckets for the shuffle, but we don't launch any reduce tasks until the shuffle is complete
map_output_refs = [count_words_in_chunk_return_bucket_refs.remote(docs) for docs in chunks]
print("Mapper[0] bucket refs:\n" + "[" + "\n ".join(map(str, map_output_refs[0])) + "]")

# The driver now shuffles refs instead of bucket payloads.
routed_bucket_refs = {0: [], 1: [], 2: []}
for result in map_output_refs:
    for reducer_id, bucket_ref in enumerate(result):
        routed_bucket_refs[reducer_id].append(bucket_ref)

# Only after the full shuffle do we launch one reducer task per bucket group, passing bucket refs as separate arguments
reduce_output_refs = [
    reduce_bucket_refs.remote(reducer_id, *routed_bucket_refs[reducer_id])
    for reducer_id in range(len(routed_bucket_refs))
]
reduce_outputs = ray.get(reduce_output_refs)

# Display reducer outputs and verify global count matches direct count by merging reducer outputs after the shuffle
print("\nReducer outputs:")
for result in reduce_outputs:
    print(f"reducer {result['reducer_id']} | unique_words ={len(result['counts']):>3} | "
          f"top words: {format_top(result['counts'], 4)}")

# Verify global count matches direct count by merging reducer outputs on the driver after the shuffle
total_counts_object_refs = Counter()
for result in reduce_outputs:
    total_counts_object_refs.update(result["counts"])

# Compare to direct count on the driver without shuffle
expected_counts = Counter(flatten_tokens(corpus))
print("\nGlobal word count (top words):")
print(format_top(total_counts_object_refs, 12))
print(f"matches direct count = {total_counts_object_refs == expected_counts}")

Launching 67 tasks...
Mapper[0] bucket refs:
[ObjectRef(25b84cb9aa5e69f5ffffffffffffffffffffffff0100000001000000)
 ObjectRef(25b84cb9aa5e69f5ffffffffffffffffffffffff0100000002000000)
 ObjectRef(25b84cb9aa5e69f5ffffffffffffffffffffffff0100000003000000)]

Reducer outputs:
reducer 0 | unique_words = 37 | top words: a:60, and:40, before:20, dawn:20
reducer 1 | unique_words = 28 | top words: markets:20, opened:20, river:20, pipeline:20
reducer 2 | unique_words = 23 | top words: the:160, while:40, winter:20, station:20

Global word count (top words):
the:160, a:60, and:40, while:40, before:20, dawn:20, beside:20, interns:20, halves:20, build:20, an:20, bicycle:20
matches direct count = True


### Anti-pattern: shuffle synchronization barrier
The improved version removes driver data materialization, but there is still a global synchronization point: each reducer needs one bucket from every mapper, so reducer work is delayed until all buckets are available in the object store.

We'll surface the problem by:
1. Creating a straggler: one very slow map task.
2. Making reduction take a small amount of time per processed bucket so the idle barrier is easier to notice.

In [20]:
MapReduceResult = tuple[int, Counter[str], Counter[str], Counter[str]]

@ray.remote(num_returns=4)
def count_words_with_delay(chunk_id: int, docs: list[list[str]]) -> MapReduceResult:
    buckets = [Counter(), Counter(), Counter()]

    for word in flatten_tokens(docs):
        buckets[reducer_for_word(word)][word] += 1

    # New: Force one slow mapper and a visible reduce cost
    slow_id = len(chunks) - 1
    if chunk_id == slow_id:
        time.sleep(1)

    # First return is a lightweight completion token for `ray.wait()`
    return (chunk_id, *buckets)

In [21]:
reduce_delay_per_bucket_s = 0.02

@ray.remote
def reduce_bucket_refs_with_delay(reducer_id: int, *buckets: Counter[str]) -> dict:
    time.sleep(reduce_delay_per_bucket_s * len(buckets))
    counts = Counter()
    for bucket in buckets:
        counts.update(bucket)
    return {
        "reducer_id": reducer_id,
        "counts": counts,
        "num_buckets": len(buckets),
        "simulated_reduce_delay_s": reduce_delay_per_bucket_s * len(buckets),
    }

In [22]:
t0 = time.perf_counter()

map_output_refs = [count_words_with_delay.remote(chunk_id, docs) for chunk_id, docs in enumerate(chunks)]

# All reducer bucket refs are routed up front in this version
pending_chunks = []
routed_bucket_refs = {0: [], 1: [], 2: []}
for result in map_output_refs:
    pending_chunks.append(result[0])
    for reducer_id, bucket_ref in enumerate(result[1:]):
        routed_bucket_refs[reducer_id].append(bucket_ref)

# `ray.wait()` below only records completion order; it does not launch early reduce work yet
mapper_completion_order = []
while pending_chunks:
    ready_ref, pending_chunks = ray.wait(pending_chunks, num_returns=1)
    ready_chunk_id = ray.get(ready_ref[0])
    mapper_completion_order.append(ready_chunk_id)

# Barrier remains: reducers are launched only after every mapper has finished
reduce_output_refs = [
    reduce_bucket_refs_with_delay.remote(reducer_id, *routed_bucket_refs[reducer_id])
    for reducer_id in range(len(routed_bucket_refs))
]
# These are scheduled for execution only after all routed_bucket_refs are materialized in the object store

reduce_outputs = ray.get(reduce_output_refs)
runtime = time.perf_counter() - t0

print("Reducer outputs:")
for result in reduce_outputs:
    print(f"reducer {result['reducer_id']} | unique_words ={len(result['counts']):>3} | "
          f"buckets={result['num_buckets']:>2} | simulated_reduce_delay={result['simulated_reduce_delay_s']:>4.2f}s | "
          f"top words: {format_top(result['counts'], 4)}")

print(f"\nmapper completion order = {mapper_completion_order}")
print(f"runtime = {runtime:0.2f}s")

Reducer outputs:
reducer 0 | unique_words = 37 | buckets=67 | simulated_reduce_delay=1.34s | top words: a:60, and:40, before:20, dawn:20
reducer 1 | unique_words = 28 | buckets=67 | simulated_reduce_delay=1.34s | top words: markets:20, opened:20, river:20, pipeline:20
reducer 2 | unique_words = 23 | buckets=67 | simulated_reduce_delay=1.34s | top words: the:160, while:40, winter:20, station:20

mapper completion order = [1, 0, 2, 3, 4, 5, 6, 8, 10, 7, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 32, 31, 33, 34, 35, 36, 37, 38, 41, 42, 40, 43, 47, 44, 46, 39, 49, 45, 50, 51, 52, 48, 53, 54, 60, 56, 55, 58, 57, 59, 61, 62, 63, 64, 65, 66]
runtime = 2.42s


### Version 3.0: partial reduce with `ray.wait()`
What changes in this version:
- `ray.wait()` gives us mapper completion order without blocking on the entire map stage.
- As soon as a mapper finishes, we route its bucket refs to reducer-specific queues.
- Once a queue is large enough, we launch a remote partial reduce immediately instead of waiting for every mapper.
- That overlaps reduce work with the tail of the map stage and replaces one large fan-in with several smaller ones.
- By the time the straggler finishes, most of the reduction work is already done.

In [23]:
@ray.remote
def partial_reduce(*buckets: Counter[str]) -> Counter[str]:
    time.sleep(reduce_delay_per_bucket_s * len(buckets))
    total = Counter()
    for part in buckets:
        total.update(part)
    return total

In [24]:
t0 = time.perf_counter()
map_output_refs = [count_words_with_delay.remote(chunk_id, docs) for chunk_id, docs in enumerate(chunks)]

# Reuse the completion token to react to mapper finishes as they happen
pending_chunks = [result[0] for result in map_output_refs]

mapper_completion_order = []
# New: keep ready buckets and launched partial reduces per reducer
ready_bucket_refs = {0: [], 1: [], 2: []}
partial_reduce_refs = {0: [], 1: [], 2: []}
while pending_chunks:
    ready_ref, pending_chunks = ray.wait(pending_chunks, num_returns=1)
    ready_chunk_id = ray.get(ready_ref[0])
    mapper_completion_order.append(ready_chunk_id)

    # Route only the bucket refs that belong to the mapper that just finished
    for reducer_id, bucket_ref in enumerate(map_output_refs[ready_chunk_id][1:]):
        ready_bucket_refs[reducer_id].append(bucket_ref)

    print(f"mapper {ready_chunk_id:>2} finished -> routed 3 bucket refs")

    for reducer_id in range(len(ready_bucket_refs)):
        if len(ready_bucket_refs[reducer_id]) >= 10:
            # Launch reduce work early instead of waiting for a full reducer input
            partial_reduce_refs[reducer_id].append(partial_reduce.remote(*ready_bucket_refs[reducer_id]))
            print(f"launched partial reduce for reducer {reducer_id} with {len(ready_bucket_refs[reducer_id])} ready buckets")
            ready_bucket_refs[reducer_id] = []

# Flush any leftovers that never reached the batch threshold
for reducer_id in range(len(ready_bucket_refs)):
    if ready_bucket_refs[reducer_id]:
        partial_reduce_refs[reducer_id].append(partial_reduce.remote(*ready_bucket_refs[reducer_id]))
        print(f"final partial reduce for reducer {reducer_id} with {len(ready_bucket_refs[reducer_id])} ready buckets")
        ready_bucket_refs[reducer_id] = []

# Final step: merge the partial results for each reducer
for reducer_id in range(len(partial_reduce_refs)):
    print(f"final merge for reducer {reducer_id} with {len(partial_reduce_refs[reducer_id])} partial results")
final_reduce_refs = [
    partial_reduce.remote(*partial_reduce_refs[reducer_id])
    for reducer_id in range(len(partial_reduce_refs))
]
final_reduce_outputs = ray.get(final_reduce_refs)
runtime = time.perf_counter() - t0
final_counts = Counter()

# Display reducer outputs and verify global count matches direct count by merging reducer outputs after the shuffle
print("Reducer outputs:")
for reducer_id, counts in enumerate(final_reduce_outputs):
    final_counts.update(counts)
    print(f"reducer {reducer_id} | unique_words={len(counts):>3} | top words: {format_top(counts, 4)}")

# Verify global count matches direct count by merging reducer outputs on the driver after the shuffle
expected_counts = Counter(flatten_tokens(corpus))
print(f"matches direct count = {final_counts == expected_counts}")
print(f"\nmapper completion order = {mapper_completion_order}")
print(f"runtime = {runtime:0.2f}s")

mapper  8 finished -> routed 3 bucket refs
mapper  6 finished -> routed 3 bucket refs
mapper  7 finished -> routed 3 bucket refs
mapper  1 finished -> routed 3 bucket refs
mapper  5 finished -> routed 3 bucket refs
mapper  3 finished -> routed 3 bucket refs
mapper  9 finished -> routed 3 bucket refs
mapper  4 finished -> routed 3 bucket refs
mapper 10 finished -> routed 3 bucket refs
mapper  2 finished -> routed 3 bucket refs
launched partial reduce for reducer 0 with 10 ready buckets
launched partial reduce for reducer 1 with 10 ready buckets
launched partial reduce for reducer 2 with 10 ready buckets
mapper 15 finished -> routed 3 bucket refs
mapper 13 finished -> routed 3 bucket refs
mapper 23 finished -> routed 3 bucket refs
mapper 17 finished -> routed 3 bucket refs
mapper 21 finished -> routed 3 bucket refs
mapper 20 finished -> routed 3 bucket refs
mapper 14 finished -> routed 3 bucket refs
mapper 11 finished -> routed 3 bucket refs
mapper 18 finished -> routed 3 bucket refs
map

## 3. Summary

We wrote three versions of MapReduce:

1. Driver shuffle with `ray.get(...)`: simple to read, but it materializes all mapper output on the driver and waits for the whole map phase before launching the reduce phase.
2. Driver `ObjectRef` routing: reducers fetch their assigned buckets from the object store. This avoids driver materialization, but reducers still wait for all mappers.
3. Partial reduce with `ray.wait()`: early map outputs are reduced in batches simultaneously with the late map tasks. This requires a small final reduce phase to merge the partial results.

## 4. Optional exercise

Write MapReduce with reducer actors that maintain and update a single word-count state instead of doing partial reductions recursively.

- keep the mapper-side bucketing and the `ray.wait()` coordination loop
- create one actor per reducer
- let each actor accept bucket updates and merge them into its internal `Counter`
- add a method that returns the final counts for that reducer
- compare the tradeoff against the partial-reduce version: statefulness, parallelism, update granularity, and fault tolerance


In [25]:
@ray.remote
class ReducerActor:
    def __init__(self, reducer_id: int) -> None:
        self.reducer_id = reducer_id
        self.counts = Counter()

    # Let each actor accept bucket updates and merge them into its internal `Counter`
    def update(self, bucket: Counter[str]) -> None:
        self.counts.update(bucket)

    # Add a method that returns the final counts for that reducer
    def get_counts(self) -> Counter[str]:
        return self.counts

In [26]:
print(f"Corpus size: {len(corpus)} docs")
print(f"Tasks to process (chunks): {len(chunks)}")
print(f"Slow mapper chunk: {len(chunks) - 1}")

# Create one actor per reducer
reducers = [ReducerActor.remote(i) for i in range(3)]

# Map phase
t0 = time.perf_counter()
map_output_refs = [count_words_with_delay.remote(i, docs) for i, docs in enumerate(chunks)]
pending_chunks = [result[0] for result in map_output_refs]

# Keep the mapper-side bucketing and the `ray.wait()` coordination loop
while pending_chunks:
    ready_ref, pending_chunks = ray.wait(pending_chunks, num_returns=1)
    chunk_id = ray.get(ready_ref[0])
    for reducer_id, bucket_ref in enumerate(map_output_refs[chunk_id][1:]):
        reducers[reducer_id].update.remote(bucket_ref)

# Get results
reduce_outputs = ray.get([r.get_counts.remote() for r in reducers])
runtime = time.perf_counter() - t0

# Compare the tradeoff against the partial-reduce version: statefulness, parallelism, update granularity, and fault tolerance
final_counts = Counter()
for i, counts in enumerate(reduce_outputs):
    final_counts.update(counts)
    print(f"reducer {i} | unique_words={len(counts):>3} | top: {format_top(counts, top_k=4)}")

print(f"Top words: {format_top(final_counts, top_k=12)}")
print(f"Correct: {final_counts == Counter(flatten_tokens(corpus))}")
print(f"Runtime: {runtime:.2f}s")

Corpus size: 200 docs
Tasks to process (chunks): 67
Slow mapper chunk: 66
reducer 0 | unique_words= 37 | top: a:60, and:40, before:20, dawn:20
reducer 1 | unique_words= 28 | top: markets:20, opened:20, river:20, pipeline:20
reducer 2 | unique_words= 23 | top: the:160, while:40, winter:20, station:20
Top words: the:160, a:60, and:40, while:40, before:20, dawn:20, beside:20, interns:20, halves:20, build:20, an:20, bicycle:20
Correct: True
Runtime: 1.42s


In [27]:
ray.shutdown()